# Restaurant Simulation

### In my model, there are participants, where each operates as a separate thread:

* 1 Kitchen - takes orders from the queue and cooks them one by one
* 2 Waiters - take orders from clients, send them to the kitchen, wait for them to be ready, and deliver the food
* 5 Clients - arrive, wait for an available waiter, place an order, wait for the food, eat, and leave

# Settings

---



In [1]:
import random
import threading
import time
from collections import deque


NUM_CLIENTS = 5
NUM_WAITERS = 2
MIN_T, MAX_T = 2, 5          # random time range in seconds

# Resourses for all

---

This part starts by setting up shared information that all parts of the program can use. This includes a special list for orders that are waiting to be picked up, a list of waiters who are free to work, and a count of how many customers have been served. To make sure that different parts of the program don't try to use this information at the same time, it uses "locks". These locks keep different parts of the program from stepping on each other's toes.

It also sets up a separate lock for printing messages to the screen so that everything shows up in the right order.

To keep everything running smoothly, it uses special tools called "semaphores". These tools can pause certain parts of the program (like waiting for a waiter to be free or waiting for an order to arrive) and then let them go when it's time. There's also a special signal that tells the program when it's done, so that everything can shut down properly.

Finally, it records the exact time when the program starts so that it can keep track of how long things take.

In [2]:
kitchen_queue = deque()      # Order queue - elements are Order objects
queue_lock = threading.Lock()

free_waiters = deque()       # Pool of available waiters
waiters_lock = threading.Lock()

served_count = 0             # Number of clients already served
counter_lock = threading.Lock()

print_lock = threading.Lock()

# "Waiting" synchronization (these are signals between threads, not data protection)
waiter_available = threading.Semaphore(0)   # +1 every time a waiter becomes available
kitchen_has_orders = threading.Semaphore(0) # +1 for every new order added to the queue
all_done = threading.Event()                # All clients served then kitchen finishes operation

START = time.time()

# Function for log print for beauty and class for creating kithen objects

---

This section explains two important tools for the simulation: keeping track of what happens and monitoring the status of orders.

1. Logging - this tool records messages from the simulation. It shows how much time has passed since the simulation started, in a format like "HH:MM:SS". It makes sure that messages from different parts of the simulation don't mix up or get cut off when displayed on the screen. The "flush=True" part makes sure that the messages appear right away.
2. Order - this tool represents a customer's order. It has information about the customer and the waiter who is serving them. Most importantly, it has a flag called "ready". When the kitchen is done cooking the food, it sets this flag. This tells the waiter that the food is ready to be delivered.


In [3]:
def log(actor, msg):
    elapsed = int(time.time() - START)
    ts = f"{elapsed // 3600:02d}:{(elapsed % 3600) // 60:02d}:{elapsed % 60:02d}"
    with print_lock:
        print(f"[{ts}] {actor}: {msg}", flush=True)


class Order:
    def __init__(self, client, waiter):
        self.client = client
        self.waiter = waiter
        self.ready = threading.Event()   # Kitchen sets this flag when the dish is cooked

# Kitchen

---

This function is like a helper that keeps the kitchen running smoothly. It keeps checking for orders using a special tool called 'kitchen_has_orders' and waits for up to half a second each time. This helps it see if all customers have been served. Once all customers are done, it stops working.

When an order comes in, it safely grabs it from a list called 'kitchen_queue'. It uses a special lock called 'queue_lock' to make sure no one else is messing with the list at the same time. Then, it simulates the cooking time by waiting for a set amount of time. Finally, it tells the waiter that the food is ready by setting a flag on the order.

In [4]:
def kitchen():
    global served_count
    log("Kitchen", "opened, waiting for orders")
    while True:
        if not kitchen_has_orders.acquire(timeout=0.5):
            if all_done.is_set():
                break
            continue

        with queue_lock:
            order = kitchen_queue.popleft()

        cook_time = random.randint(MIN_T, MAX_T)
        log("Kitchen", f"cooking order for {order.client} (~{cook_time}s)")
        time.sleep(cook_time)
        log("Kitchen", f"order for {order.client} is ready")
        order.ready.set()
    log("Kitchen", "closed")

# Waiters

---

This part of the program makes a loop for a waiter. It keeps checking a special list for tasks. When it finds a task, the waiter acts like they're taking an order from a customer. They add the order to a queue for the kitchen, using a special lock to make sure only one person can add to the queue at a time. Then, the waiter tells the kitchen that there's an order by using a signal.

After that, the waiter waits until the kitchen tells them the food is ready. When the kitchen says it's ready, the waiter acts like they're bringing the food to the customer, tells the customer that their food is here, and puts their name back on a list of available waiters. They also increase a count of available waiters so that other customers can get served.

In [5]:
def waiter(name, orders_from_clients):
    my_inbox = orders_from_clients[name]
    while True:
        client_ev = my_inbox.get()
        if client_ev is None:
            break
        client_name, served_event = client_ev

        log(name, f"taking order from {client_name}")
        time.sleep(random.uniform(1, 2))

        order = Order(client_name, name)

        with queue_lock:
            kitchen_queue.append(order)
            qlen = len(kitchen_queue)

        kitchen_has_orders.release()
        log(name, f"sent order of {client_name} to kitchen (queue length: {qlen})")

        order.ready.wait()

        log(name, f"delivering food to {client_name}")
        time.sleep(random.randint(1, 2))
        served_event.set()


        with waiters_lock:
            free_waiters.append(name)

        waiter_available.release()
        log(name, "is free again")

# Clients

---

This function controls how a client works in a restaurant simulation. When a client arrives at a random time, they wait for a signal from 'waiter_available'. Once they get the signal, they safely pick a free waiter from a list of available waiters. The client gives the waiter a task and then waits for the meal to be delivered. After eating for a random amount of time, the client safely increases a count of how many meals have been served. If this client is the last one, they signal that the restaurant can close down.

In [6]:
def client(name, inboxes):
    global served_count
    time.sleep(random.uniform(0, 4))
    log(name, "arrived")

    waiter_available.acquire()

    with waiters_lock:
        w = free_waiters.popleft()

    log(name, f"assigned to {w}")

    served_event = threading.Event()
    inboxes[w].put((name, served_event))
    served_event.wait()

    eat_time = random.randint(MIN_T, MAX_T)
    log(name, f"is eating (~{eat_time}s)")
    time.sleep(eat_time)
    log(name, "finished and left")

    with counter_lock:
        served_count += 1
        current = served_count
        if current == NUM_CLIENTS:
            all_done.set()

    log(name, f"(served so far: {current}/{NUM_CLIENTS})")

# Main function

---

The main function is like the boss in a restaurant simulation. It sets everything up and makes sure everything runs smoothly. First, it creates separate boxes for each waiter's tasks, using safe ways to handle multiple tasks at once. It also makes sure there are enough waiters available.

Next, it starts different threads for the kitchen, two waiters, and five customers. All these threads start at the same time, so they can work together. The main thread then waits for all the customers to finish eating and for the kitchen to close down.

To stop the waiters from working forever, it puts special "stop" messages in their task boxes. Once all the waiters are done and have stopped working, it records that all customers have been served and the restaurant has closed.

In [7]:
def main():
    import queue
    waiter_names = [f"Waiter-{i}" for i in range(1, NUM_WAITERS + 1)]
    inboxes = {w: queue.Queue() for w in waiter_names}

    for w in waiter_names:
        free_waiters.append(w)
        waiter_available.release()

    threads = [threading.Thread(target=kitchen, name="Kitchen")]
    threads += [threading.Thread(target=waiter, args=(w, inboxes), name=w)
                for w in waiter_names]
    threads += [threading.Thread(target=client, args=(f"Client-{i}", inboxes),
                                 name=f"Client-{i}")
                for i in range(1, NUM_CLIENTS + 1)]

    log("Main", "restaurant opens")
    for t in threads:
        t.start()

    for t in threads:
        if t.name.startswith("Client") or t.name == "Kitchen":
            t.join()

    for w in waiter_names:
        inboxes[w].put(None)
    for t in threads:
        if t.name.startswith("Waiter"):
            t.join()

    log("Main", f"restaurant closed, all {served_count} clients served")


if __name__ == "__main__":
    main()

[00:00:00] Main: restaurant opens
[00:00:00] Kitchen: opened, waiting for orders
[00:00:00] Client-4: arrived
[00:00:00] Client-4: assigned to Waiter-1
[00:00:00] Waiter-1: taking order from Client-4
[00:00:00] Client-3: arrived
[00:00:00] Client-3: assigned to Waiter-2
[00:00:00] Waiter-2: taking order from Client-3
[00:00:02] Client-1: arrived
[00:00:02] Waiter-1: sent order of Client-4 to kitchen (queue length: 1)
[00:00:02] Kitchen: cooking order for Client-4 (~2s)
[00:00:02] Waiter-2: sent order of Client-3 to kitchen (queue length: 1)
[00:00:02] Client-2: arrived
[00:00:03] Client-5: arrived
[00:00:04] Kitchen: order for Client-4 is ready
[00:00:04] Kitchen: cooking order for Client-3 (~2s)
[00:00:04] Waiter-1: delivering food to Client-4
[00:00:06] Kitchen: order for Client-3 is ready
[00:00:06] Client-4: is eating (~5s)
[00:00:06] Waiter-1: is free again
[00:00:06] Waiter-2: delivering food to Client-3
[00:00:06] Client-1: assigned to Waiter-1
[00:00:06] Waiter-1: taking order 